# User anomaly and hybrid-weight sensitivity

Privacy rule: no subject, body, preview, or attachment content is loaded.

In [ ]:
PROJECT_ROOT = '..'
RANDOM_SEED = 365

In [ ]:
from pathlib import Path
ROOT = Path(PROJECT_ROOT).resolve()
DATA_ROOT = ROOT / 'data'
ARTIFACT_ROOT = ROOT / 'artifacts/models/current'

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
daily = pd.read_csv(DATA_ROOT/'synthetic/m365_user_daily.csv.gz', parse_dates=['date'])
sample = daily[daily.user_id == daily.user_id.iloc[0]]
sns.lineplot(data=sample, x='date', y='behaviour_anomaly_score', hue='label'); plt.title('Behavior anomaly timeline')

In [ ]:
from sklearn.ensemble import IsolationForest
baseline = daily[daily.date < daily.date.quantile(.7)].copy(); values = baseline[['emails_received','external_sender_ratio','sender_domain_novelty']].fillna(0)
median = values.median(); mad = (values-median).abs().median().replace(0,1); baseline['robust_z'] = ((values-median)/mad).abs().max(axis=1)
iso = IsolationForest(contamination=.03,random_state=RANDOM_SEED).fit(values); baseline['isolation_score'] = -iso.score_samples(values)
fig,axes=plt.subplots(1,2,figsize=(12,4)); sns.histplot(baseline,x='robust_z',hue='label',ax=axes[0]); sns.histplot(baseline,x='isolation_score',hue='label',ax=axes[1]); plt.tight_layout()

In [ ]:
from packages.ml.m365risk_ml.scoring import HYBRID_WEIGHTS
weights = pd.Series(HYBRID_WEIGHTS); weights.plot.bar(title='PDF hybrid weights', color='#4f8cff'); plt.ylim(0, .4)

In [ ]:
entra = daily.entra_risk_level.map({'none':0,'low':.3,'medium':.65,'high':1}).fillna(0); posture=(~daily.is_mfa_registered).astype(float); privilege=daily.is_admin.astype(float)
rows=[]
for email_weight in np.linspace(.2,.55,8):
    remaining=1-email_weight; score=email_weight*daily.email_risk_max + remaining*(.31*daily.behaviour_anomaly_score+.31*entra+.23*posture+.15*privilege)
    top=score>=score.quantile(.9); rows.append({'email_weight':email_weight,'top_10_recall':(top & daily.label.eq(1)).sum()/daily.label.sum(),'false_alerts_per_100':100*(top & daily.label.eq(0)).mean()})
sensitivity=pd.DataFrame(rows); sensitivity.plot(x='email_weight',secondary_y='false_alerts_per_100',marker='o',title='Weight sensitivity'); display(sensitivity)

In [ ]:
base_score=.35*daily.email_risk_max+.20*daily.behaviour_anomaly_score+.20*entra+.15*posture+.10*privilege
top_k=[]
for fraction in np.linspace(.01,.25,25):
    cutoff=base_score.quantile(1-fraction); selected=base_score>=cutoff
    top_k.append({'review_fraction':fraction,'compromise_recall':(selected & daily.label.eq(1)).sum()/daily.label.sum(),'false_review_rate':(selected & daily.label.eq(0)).sum()/max(1,daily.label.eq(0).sum())})
top_k=pd.DataFrame(top_k); top_k.plot(x='review_fraction', y=['compromise_recall','false_review_rate'], marker='o', title='Top-k detection and review trade-off'); plt.axvline(.10,color='red',ls='--'); plt.tight_layout(); display(top_k[top_k.review_fraction.between(.09,.11)])